In [0]:

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

customer_df=spark.read.parquet("/Volumes/workspace/default/shree_project/silver/customers")
order_df=spark.read.parquet("/Volumes/workspace/default/shree_project/silver/orders")
product_df=spark.read.parquet("/Volumes/workspace/default/shree_project/silver/products")
payment_df=spark.read.parquet("/Volumes/workspace/default/shree_project/silver/payments")

customer_df.show()
product_df.show()
payment_df.show()


customer_dim=(
    customer_df.dropDuplicates(['customer_id'])
    .filter(col('customer_id').isNotNull())
)

customer_dim.write \
  .format("delta") \
  .mode("overwrite") \
  .save("/Volumes/workspace/default/shree_project/gold/customer_dim")

product_dim=(
    product_df.dropDuplicates(['product_id'])\
        .filter(col('product_id').isNotNull())
        
)

product_dim.write \
  .format("delta") \
  .mode("overwrite") \
  .save("/Volumes/workspace/default/shree_project/gold/product_dim")


payment_dim=(
    payment_df.dropDuplicates(['payment_id'])\
        .filter(col('order_id').isNotNull())
)

payment_dim.write \
  .format("delta") \
  .mode("overwrite") \
  .save("/Volumes/workspace/default/shree_project/gold/payment_dim")



order_fact= (
    order_df.join(
        customer_dim,"customer_id","inner")\
            .join(product_dim,"product_id","inner")\
                .select(

                    order_df.order_id,
                    order_df.order_date,
                    order_df.customer_id,
                    order_df.product_id,
                    order_df.quantity,
                    order_df.order_status,
                    product_dim.price.alias("unit_price"),
                    (order_df.quantity*product_dim.price).alias("order_amount")

                        )
    )


order_fact.write \
  .format("delta") \
  .mode("overwrite") \
  .save("/Volumes/workspace/default/shree_project/gold/order_fact")

    
